# Linux inside Jupyter — practice notebook
**Lecture 3.1 · Modern Data Architectures for Big Data I · IE University**

Run each cell in order (Shift + Enter). In a notebook:
- `!command` runs a Linux command in a new shell
- `var = !command` saves its output as a Python list
- `{python_var}` puts a Python value inside a command
- `%cd` changes the notebook's folder (`!cd` does **not** stick)
- `%%bash` at the top of a cell runs the whole cell as one bash script

## 1. Where am I and what's here?

In [2]:
! pwd
! whoami
! ls -lah

/home/basqueteer/Projects/iet/modern-data-architectures-i/session-02-osbdet-and-foundations
basqueteer
total 1.7M
drwxrwxr-x  2 basqueteer basqueteer 4.0K Oct 16 16:18 .
drwxrwxr-x 16 basqueteer basqueteer 4.0K Oct 16 16:20 ..
-rw-rw-r--  1 basqueteer basqueteer 441K Oct 15 22:56 lecture-02-01-course-environment-slides.html
-rw-rw-r--  1 basqueteer basqueteer  35K Oct 15 22:56 lecture-02-01-course-environment-slides.zip
-rw-rw-r--  1 basqueteer basqueteer  17K Oct 15 22:58 lecture-02-01-course-environment.md
-rw-rw-r--  1 basqueteer basqueteer 586K Oct 16 16:16 lecture-02-02-linux-commands-slides.html
-rw-rw-r--  1 basqueteer basqueteer  13K Oct 16 16:16 lecture-02-02-linux-commands.md
-rw-rw-r--  1 basqueteer basqueteer 530K Oct 16 16:16 lecture-02-03-git-github-essentials-slides.html
-rw-rw-r--  1 basqueteer basqueteer  13K Oct 16 16:16 lecture-02-03-git-github-essentials.md
-rw-rw-r--  1 basqueteer basqueteer 7.3K Oct 16 16:22 linux-in-jupyter-practice.ipynb


## 2. Get some data (MovieLens small)
If the download fails (no internet), copy `ml-latest-small.zip` into this folder by hand.

In [3]:
! curl -sO https://files.grouplens.org/datasets/movielens/ml-latest-small.zip
! unzip -oq ml-latest-small.zip
! ls -lh ml-latest-small

total 3.2M
-rw-r--r-- 1 basqueteer basqueteer 8.2K Sep 26  2018 README.txt
-rw-r--r-- 1 basqueteer basqueteer 194K Sep 26  2018 links.csv
-rw-r--r-- 1 basqueteer basqueteer 483K Sep 26  2018 movies.csv
-rw-r--r-- 1 basqueteer basqueteer 2.4M Sep 26  2018 ratings.csv
-rw-r--r-- 1 basqueteer basqueteer 116K Sep 26  2018 tags.csv


## 3. Look inside files

In [4]:
! head -n 3 ml-latest-small/movies.csv
! tail -n 2 ml-latest-small/movies.csv
! wc -l ml-latest-small/*.csv

movieId,title,genres
1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
2,Jumanji (1995),Adventure|Children|Fantasy
193587,Bungo Stray Dogs: Dead Apple (2018),Action|Animation
193609,Andrew Dice Clay: Dice Rules (1991),Comedy
   9743 ml-latest-small/links.csv
   9743 ml-latest-small/movies.csv
 100837 ml-latest-small/ratings.csv
   3684 ml-latest-small/tags.csv
 124007 total


## 4. Search and count

In [5]:
! grep -i "toy story" ml-latest-small/movies.csv
! grep -c "Comedy" ml-latest-small/movies.csv

1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
3114,Toy Story 2 (1999),Adventure|Animation|Children|Comedy|Fantasy
78499,Toy Story 3 (2010),Adventure|Animation|Children|Comedy|Fantasy|IMAX
3756


## 5. A pipeline: top 5 genres (map → sort → reduce on one machine)

In [6]:
! tail -n +2 ml-latest-small/movies.csv | awk -F, '{print $NF}' | tr -d '\r' | tr '|' '\n' | sort | uniq -c | sort -nr | head -5

   4361 Drama
   3756 Comedy
   1894 Thriller
   1828 Action
   1596 Romance


## 6. Shell output → Python, Python → shell

In [10]:
files = ! ls ml-latest-small/*.csv
print(len(files), 'files:', files)

4 files: ['ml-latest-small/links.csv', 'ml-latest-small/movies.csv', 'ml-latest-small/ratings.csv', 'ml-latest-small/tags.csv']


In [9]:
for dataset in ['movies', 'ratings']:
    n = ! wc -l < ml-latest-small/{dataset}.csv
    print(f'{dataset}: {int(n[0]) - 1:,} rows (without header)')

movies: 9,742 rows (without header)
ratings: 100,836 rows (without header)


## 7. The `!cd` trap

In [11]:
! cd ml-latest-small
! pwd   # still the same folder!

/home/basqueteer/Projects/iet/modern-data-architectures-i/session-02-osbdet-and-foundations


In [12]:
%cd ml-latest-small
! ls
%cd ..

/home/basqueteer/Projects/iet/modern-data-architectures-i/session-02-osbdet-and-foundations/ml-latest-small
README.txt  links.csv  movies.csv  ratings.csv	tags.csv
/home/basqueteer/Projects/iet/modern-data-architectures-i/session-02-osbdet-and-foundations


## 8. A whole cell in bash: build a landing zone

In [13]:
%%bash
mkdir -p landing/movielens/{movies,ratings,tags,links}
for d in movies ratings tags links; do
  cp ml-latest-small/$d.csv landing/movielens/$d/
done
find landing -type f | sort
du -sh landing

landing/movielens/links/links.csv
landing/movielens/movies/movies.csv
landing/movielens/ratings/ratings.csv
landing/movielens/tags/tags.csv
3.2M	landing


## 9. Your turn
1. How many ratings are exactly `5.0`? (hint: `cut -d, -f3`, `grep -c`)
2. Which file in `ml-latest-small` is the biggest? (hint: `ls -S`)
3. Copy `tags.csv` into a new folder `backup/` and check it with `ls -l backup`

In [ ]:
# 1.

In [ ]:
# 2.

In [ ]:
# 3.

## 10. Clean up (optional)

In [ ]:
# !rm -r landing backup ml-latest-small ml-latest-small.zip